# Clase 2 — ASR con Whisper y síntesis de voz (TTS)

## Pregunta central

> **¿Cómo convertimos voz en texto y texto en voz?**

## Idea principal

El reconocimiento automático de voz (ASR) transforma audio en texto; la síntesis de voz (TTS) hace el camino inverso. Juntos forman la base de un asistente conversacional: el usuario habla, el sistema entiende, responde y "habla" de vuelta.

## Objetivos de aprendizaje

Al finalizar la clase deberías poder:

- Explicar el pipeline de ASR (audio → features → modelo → texto).
- Transcribir un audio con Whisper usando `transformers`.
- Sintetizar voz en español con un modelo TTS local.
- Armar un pipeline integrado voz → texto → respuesta → voz.
- Reconocer los límites y riesgos de privacidad del audio en salud.

## Recorrido de la clase

| Paso | Tema |
|---:|---|
| 1 | Qué es ASR y cómo funciona |
| 2 | Transcribir con Whisper |
| 3 | Qué es TTS y cómo funciona |
| 4 | Sintetizar voz en español |
| 5 | Pipeline integrado: asistente de consultorio |
| 6 | Límites y privacidad |
| 7 | Actividad: mini asistente |

## Cómo trabajar con este notebook

1. Ejecutá las celdas en el orden propuesto.
2. Antes de modificar código, observá y describí el resultado.
3. Cambiá solamente las variables marcadas con `TODO`.
4. La primera ejecución descarga los modelos (Whisper Tiny y MMS-TTS). Después quedan en caché.
5. Si aparece un término nuevo, buscá primero su definición en el glosario de la clase.

**Conexión con el programa:** este par ASR + TTS es el esqueleto de los agentes conversacionales que agendan turnos y gestionan recetas en el Track Salud.


## Glosario mínimo

| Término | Explicación breve |
|---|---|
| ASR | Reconocimiento automático de voz: audio → texto |
| TTS | Síntesis de voz: texto → audio |
| Whisper | Modelo de ASR de OpenAI, disponible en varios tamaños |
| Whisper Tiny | Versión más chica de Whisper (~39M parámetros) |
| Pipeline | Función de `transformers` que une modelo + pre/postprocesamiento |
| Transcripción | Texto que resulta de convertir audio en palabras |
| Alucinación | Texto inventado por el modelo cuando no está seguro |
| Sample rate | Muestras por segundo; Whisper espera 16 kHz |
| MMS-TTS | Modelo de síntesis de voz multilingüe de Meta |
| Latencia | Tiempo entre entrada y salida |
| Consentimiento | Permiso explícito para usar datos (clave en salud) |


---
## 1. Qué es ASR y cómo funciona

El **reconocimiento automático de voz (ASR)** convierte una señal de audio en texto.

> **Analogía del transcriptor.** Un ASR es como un transcriptor que escucha una grabación y escribe lo que escucha. Pero en vez de oídos, usa un modelo entrenado con muchísimas horas de audio y texto.

El pipeline tiene varias etapas:

```text
audio (waveform)
    |
    v
features (espectrograma / mel)
    |
    v
modelo (red neuronal)
    |
    v
texto transcrito
```

> **Pensalo así:** en la clase 1 vimos cómo convertir audio en espectrograma. Whisper hace eso internamente y luego usa una red neuronal para "leer" ese espectrograma y producir texto.

### Errores típicos del ASR

| Error | Qué pasa | Ejemplo |
|---|---|---|
| Homófonos | Palabras que suenan igual | "haya" vs "halla" |
| Ruido de fondo | El modelo se confunde | Música de espera transcrita |
| Alucinación | Inventa texto | Silencio → frase inventada |
| Nombres propios | Difíciles de transcribir | Apellidos poco comunes |

> **Importante:** la transcripción no es perfecta. En salud, un error puede cambiar el sentido de un dato. Por eso siempre hay que validar antes de usar el texto para tomar decisiones.


---
## 2. Transcribir con Whisper

Vamos a usar **Whisper Tiny**, la versión más chica de Whisper. Es lo suficientemente liviana para correr en CPU y alcanza para experimentar.

> **Pensalo así:** Whisper viene en tamaños (tiny, base, small, medium, large). Más grande = más preciso pero más lento y pesado. Para una clase en CPU, tiny es el equilibrio correcto.

La primera ejecución descarga el modelo (~75 MB). Después queda en caché.


In [5]:
# --- Cargar el pipeline de ASR ---
import os
import numpy as np
import soundfile as sf
from transformers import pipeline

# Ruta del audio de ejemplo.
RUTA_AUDIO = os.path.join("..", "assets", "audio", "curso_ia_es.wav")

# Creamos el pipeline de reconocimiento de voz con Whisper Tiny.
# device=0 usaría GPU; dejamos CPU por defecto.
asr = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-tiny",
    device=-1,
)

print("Pipeline ASR listo (Whisper Tiny).")


Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

Pipeline ASR listo (Whisper Tiny).


### Transcribir el audio

El pipeline acepta la ruta del archivo o el arreglo de muestras. Le pasamos la ruta y el sample rate correcto (16 kHz).


In [7]:
# --- Transcribir el audio de ejemplo ---
resultado = asr(RUTA_AUDIO)

print("Transcripción de Whisper:")
print("-" * 50)
print(resultado["text"])
print("-" * 50)


Transcripción de Whisper:
--------------------------------------------------
 La inteligencia artificial aprende patrones a partir de datos.
--------------------------------------------------


---
## 3. Qué es TTS y cómo funciona

La **síntesis de voz (TTS)** convierte texto en audio hablado.

> **Analogía del locutor.** Un TTS es como un locutor que lee un guion. Pero en vez de una persona, es un modelo que genera la forma de onda del habla a partir del texto.

```text
texto
    |
    v
modelo TTS
    |
    v
audio (waveform)
    |
    v
archivo WAV
```

> **Pensalo así:** el ASR "lee" audio y escribe texto; el TTS "lee" texto y escribe audio. Son espejos.

### Usos en salud

| Uso | Ejemplo |
|---|---|
| Confirmar turnos | "Su turno es el jueves a las 10" |
| Leer recetas | "Tome una pastilla cada 8 horas" |
| Accesibilidad | Leer indicaciones a pacientes con dificultad visual |
| Recordatorios | Llamadas automáticas de recordatorio |


---
## 4. Sintetizar voz en español

Vamos a usar **MMS-TTS** de Meta, un modelo multilingüe que incluye español. Es chico (~100 MB) y corre en CPU.

> **Pensalo así:** MMS-TTS es como un locutor multilingüe. Le decís el idioma (español) y el texto, y genera el audio.


In [9]:
# --- Cargar el pipeline de TTS ---
tts = pipeline(
    "text-to-speech",
    model="facebook/mms-tts-spa",
    device=-1,
)

print("Pipeline TTS listo (MMS-TTS español).")


Loading weights:   0%|          | 0/762 [00:00<?, ?it/s]

Pipeline TTS listo (MMS-TTS español).


### Sintetizar una frase

El pipeline TTS devuelve un objeto con el audio (arreglo numpy) y el sample rate.


In [ ]:
# --- Sintetizar una frase de ejemplo ---
frase = "Su turno es el jueves a las diez de la mañana."

salida_tts = tts(frase)

# El resultado tiene el audio y el sample rate.
audio_tts = salida_tts["audio"]
sr_tts = salida_tts["sampling_rate"]

print(f"Audio generado: {len(audio_tts):,} muestras a {sr_tts} Hz")
print(f"Duración: {len(audio_tts) / sr_tts:.2f} s")


In [ ]:
# --- Guardar el audio sintetizado como WAV ---
import os

# Carpeta de salida para los audios generados.
os.makedirs("salidas", exist_ok=True)
ruta_tts = os.path.join("salidas", "respuesta_tts.wav")
sf.write(ruta_tts, audio_tts, sr_tts)

print("Audio guardado en:", ruta_tts)


### Escuchar y verificar

> **Pregunta de interpretación:** ¿La voz suena natural? ¿Se entiende la frase? ¿Qué limitaciones notás (entonación, velocidad, pausas)?

> **Pensalo así:** el TTS genera audio que podés guardar y reproducir. En una app, ese audio se reproduce al usuario. La calidad depende del modelo y del texto.


In [ ]:
# --- Visualizar la waveform del audio sintetizado ---
import matplotlib.pyplot as plt

tiempo_tts = np.arange(len(audio_tts)) / sr_tts
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(tiempo_tts, audio_tts, color="tab:green", lw=0.5)
ax.set_title("Waveform del audio sintetizado (TTS)")
ax.set_xlabel("Tiempo (s)")
ax.set_ylabel("Amplitud")
ax.set_xlim(0, len(audio_tts) / sr_tts)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


---
## 5. Pipeline integrado: asistente de consultorio

Ahora unimos todo: el usuario "habla" (audio), el sistema transcribe (ASR), genera una respuesta (texto) y la "lee" (TTS).

```text
audio del usuario
    |
    v
ASR (Whisper) -> texto
    |
    v
lógica de la app -> respuesta
    |
    v
TTS (MMS) -> audio de respuesta
    |
    v
reproducción al usuario
```

> **Pensalo así:** este es el esqueleto de un asistente conversacional. En las clases 5-7 agregaremos el "cerebro" (LLM) que decide la respuesta. Acá la respuesta es una regla simple.


In [ ]:
# --- Función que arma el pipeline completo ---
def asistente_voz(ruta_audio, respuesta_texto):
    """Transcribe un audio, arma una respuesta y la convierte en voz."""
    # 1) ASR: audio -> texto
    transcripcion = asr(ruta_audio)["text"]

    # 2) Lógica simple: acá iría el LLM en clases futuras.
    #    Por ahora usamos la respuesta que pasamos como argumento.

    # 3) TTS: texto -> audio
    salida = tts(respuesta_texto)

    return {
        "transcripcion": transcripcion,
        "respuesta": respuesta_texto,
        "audio": salida["audio"],
        "sr": salida["sampling_rate"],
    }

# Probamos el pipeline con el audio de ejemplo.
resultado_asistente = asistente_voz(
    RUTA_AUDIO,
    "Su turno quedó confirmado para el jueves a las diez.",
)

print("Transcripción del usuario:", resultado_asistente["transcripcion"])
print("Respuesta del asistente:  ", resultado_asistente["respuesta"])
print(f"Audio de respuesta: {len(resultado_asistente['audio']):,} muestras")


In [ ]:
# --- Guardar la respuesta del asistente ---
ruta_respuesta = os.path.join("salidas", "respuesta_asistente.wav")
sf.write(
    ruta_respuesta,
    resultado_asistente["audio"],
    resultado_asistente["sr"],
)
print("Respuesta del asistente guardada en:", ruta_respuesta)


---
## 6. Límites y privacidad

En salud, el audio es un dato sensible. Antes de construir un asistente, hay que considerar:

| Riesgo | Qué implica | Mitigación |
|---|---|---|
| Errores de transcripción | Un dato mal transcrito cambia el sentido | Validación humana en decisiones críticas |
| Datos clínicos sensibles | El audio puede contener información de salud | Consentimiento explícito |
| Almacenamiento | Guardar audio es guardar datos personales | Anonimizar, retener lo mínimo |
| Alucinaciones | El modelo inventa texto | No usar la transcripción como verdad absoluta |
| Latencia | El usuario espera respuesta | Optimizar modelos o usar streaming |

> **Importante:** un asistente de consultorio puede **agendar turnos** (bajo riesgo) pero no debería **diagnosticar** sin supervisión. El nivel de riesgo define cuánta automatización es aceptable.

> **Pensalo así:** el ASR + TTS es la "voz" del sistema. La "decisión" (qué responder) es otra capa, y ahí es donde más cuidado hay que tener en salud.


---
## 7. Actividad — mini asistente de consultorio

Armá tu propio mini asistente que:

1. Transcriba el audio de ejemplo.
2. Detecte una palabra clave en la transcripción (ej: "turno", "receta", "horario").
3. Responda con una frase según la palabra detectada.
4. Convierta la respuesta en voz y la guarde.

> **Cómo pensarlo:** la detección de palabra clave es una regla simple (búsqueda de texto). En las clases 5-7 veremos cómo un LLM hace esto de forma más flexible. Acá el objetivo es cerrar el loop completo de voz.


In [ ]:
# ✏️ PASO 1: Definí las respuestas según palabra clave.
# TODO: cambiá las respuestas o agregá más palabras clave.
RESPUESTAS = {
    "turno": "Su turno quedó confirmado para el jueves a las diez.",
    "receta": "Su receta está lista para retirar en farmacia.",
    "horario": "El consultorio atiende de lunes a viernes de ocho a dieciséis.",
    "default": "¿En qué puedo ayudarle?",
}

print("Respuestas definidas:", list(RESPUESTAS.keys()))


In [ ]:
# ✏️ PASO 2: Detectá la palabra clave y respondé.
def detectar_respuesta(transcripcion, respuestas):
    texto = transcripcion.lower()
    for clave, respuesta in respuestas.items():
        if clave in texto:
            return respuesta
    return respuestas["default"]

# Podés usar la transcripción real del audio...
# transcripcion_a_usar = resultado_asistente["transcripcion"]

# ...o un texto de ejemplo que contenga una palabra clave.
# TODO: cambiá este texto para probar otras palabras clave.
transcripcion_a_usar = "Quiero pedir un turno para el viernes"

respuesta_elegida = detectar_respuesta(transcripcion_a_usar, RESPUESTAS)
print("Transcripción:", transcripcion_a_usar)
print("Respuesta elegida:", respuesta_elegida)


In [ ]:
# ✏️ PASO 3: Convertí la respuesta en voz y guardala.
salida_final = tts(respuesta_elegida)
ruta_final = os.path.join("salidas", "respuesta_final.wav")
sf.write(ruta_final, salida_final["audio"], salida_final["sampling_rate"])

print("Respuesta final guardada en:", ruta_final)
print(f"Duración: {len(salida_final['audio']) / salida_final['sampling_rate']:.2f} s")


### Tabla de reflexión

| Pregunta | Respuesta |
|---|---|
| ¿Qué palabra clave detectó tu asistente? | |
| ¿Qué pasa si el usuario dice algo sin palabras clave? | |
| ¿Cómo cambiarías la lógica para que sea más flexible? | |
| ¿Qué riesgo de privacidad ves en guardar el audio del usuario? | |

> **Cierre:** con Whisper (ASR) y MMS-TTS armaste un asistente que escucha, entiende una intención simple y responde con voz. El mismo esqueleto, con un LLM en el medio, es la base de los agentes conversacionales de las clases 5-7.


---

## Síntesis de la clase

- El ASR convierte audio en texto; el TTS convierte texto en audio.
- Whisper Tiny transcribe en español y corre en CPU.
- MMS-TTS sintetiza voz en español de forma local.
- El pipeline voz → texto → respuesta → voz es el esqueleto de un asistente.
- En salud, los errores de transcripción y la privacidad del audio son críticos.

## Comprobación conceptual

Antes de continuar, intentá responder sin mirar el notebook:

1. ¿Cuál era el problema central de la clase?
2. ¿Qué entrada recibió el sistema y qué salida produjo?
3. ¿Qué decisión humana siguió siendo necesaria?
4. ¿Qué limitación observaste en el experimento?

Si podés explicarlo con tus propias palabras y justificarlo con un resultado visible, alcanzaste el objetivo introductorio.

## Puente con la próxima clase

La clase 3 abre el NLP clínico: tokenización, embeddings de texto y extracción de entidades. Ahí veremos cómo el texto que produce el ASR se convierte en datos estructurados (ej: "el paciente tiene diabetes" → entidad clínica).

## Conexión con el track

Salud usará ASR + TTS para asistentes conversacionales, y el NLP de la clase 3 para entender el texto transcrito.

La implementación profunda, el trabajo con datasets reales y las decisiones de producción se desarrollarán en los módulos especializados.
